# Fyre opp Neo4j
Sjekke at vi kan kjøre Neo4J

Starte med
```
sudo apt install podman
```

In [191]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_dbms_security_procedures_unrestricted=apoc.* \
    -e NEO4J_dbms_security_procedures_allowlist=apoc.* \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -e NEO4J_PLUGINS='["apoc", "gds"]' \
    -d docker.io/library/neo4j:latest

08f7ff99a143e1ee4f14fad2bcd7a7473450d93de7f870ec562e5a40b6e539e3


Når man er ferdig er det bare å kopiere identifikatoren over inn i neste kall

In [163]:
%%bash
podman kill d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d

d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d


Gi databasen litt tid til å starte

In [192]:
from neo4j import GraphDatabase

# 1. Define connection details
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connection Successful!")

Connection Successful!


Sjekke at vi har APOC tilgjengelig

In [193]:
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")



Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 137ms
	Å konsumere: 5ms


Har vi GDS

In [194]:
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Server Address: 127.0.0.1:7687
Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


Ofte vil vi lage ting i Networkx og så laste det inn slik at andre kan bruke det.

In [207]:
import gzip
import networkx as nx

with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    G = nx.read_edgelist(fd, create_using=nx.DiGraph())
#
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

nx.set_node_attributes(G, ":Person", name="labels")
nx.set_edge_attributes(G, "EPOST", name="label")

# Skriv ut grafen
nx.write_graphml(G, "neo4j/import/large_graph.graphml", named_key_ids=True)
print("ok")

Nodes: 57194
Edges: 103083


Les inn

In [208]:
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("large_graph.graphml", {storeNodeIds: true, readLabels: true})""")
for record in records:
    # Converts the entire record into a Python dict
    record_dict = record.data()
#
for k in keys:
    print(f"\t{k}: {record_dict[k]}")

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: large_graph.graphml
	source: file
	format: graphml
	nodes: 57194
	relationships: 103083
	properties: 0
	time: 1637
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 64ms
	Å konsumere: 1639ms


Sjekke en velkjent node:

For å lete etter klikker skal vi først merke noder vi ikke er interessert i

In [210]:
# Legg inn
records, summary, keys = driver.execute_query(
    """
MATCH (n:Person)
SET n.antallKanter = COUNT { (n)--() }
""")
for r in records:
    innhold = r.data()
    print(innhold)
    break
#
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.antallKanter);
""")
for r in records:
    innhold = r.data()
    print(innhold)
    break
#
print("ok")

ok


In [211]:
records, summary, keys = driver.execute_query(
    """MATCH p=()-->(:Person {id:"6"}) RETURN p;
""")
# Returnerer et sett av STIER som ender i noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er tilstrekkelig
#
records, summary, keys = driver.execute_query(
    """MATCH (p)-->(:Person {id:"6"}) RETURN p;
""")
# Returnerer et sett av NODER som har relasjoner til noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
    

{'p': [{'id': '37328', 'antallKanter': 10}, 'EPOST', {'id': '6', 'antallKanter': 263}]}
{'p': {'id': '37328', 'antallKanter': 10}}


Så leter vi etter den største klikken (som vi vet skal ha 6 elementer)
For å slette en eksisterende projeksjon:
```
CALL gds.graph.drop('dense-subgraph') YIELD graphName
```


La en projeksjon med alle noder som har mer enn fem kanter.
`undirectedRelationshipTypes: ['*']` betyr at retningen ikke teller.

In [212]:
# Start med å lage en projeksjon inn i memory.
# Først velge ut nodene, og så kopiere dem
records, summary, keys = driver.execute_query(
    """
MATCH (n) WHERE n.antallKanter >= 5
OPTIONAL MATCH (n)-[r]->(m) WHERE m.antallKanter >= 5
WITH gds.graph.project(
  'dense-subgraph',   // Graph name
  n,                  // Source nodes
  m,                  // Target nodes (from the match)
  {
    sourceNodeLabels: labels(n),
    targetNodeLabels: labels(m),
    relationshipType: type(r)
  },
  { 
    undirectedRelationshipTypes: ['*'] 
  }
) AS g
RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS rels
    """)
for record in records:
    # Converts the entire record into a Python dict
    record_dict = record.data()
#
for k in keys:
    print(f"\t{k}: {record_dict[k]}")

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	graph: dense-subgraph
	nodes: 4895
	rels: 63246
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 75ms
	Å konsumere: 264ms


Nå kan vi be om "tette" steder i grafen

In [216]:
# Start med å lage en projeksjon inn i memory.
# Først velge ut nodene, og så kopiere dem
records, summary, keys = driver.execute_query(
    """
CALL gds.kcore.stream('dense-subgraph')
YIELD nodeId, coreValue
RETURN gds.util.asNode(nodeId).id AS sender, coreValue
ORDER BY sender DESC
    """)
print(keys)
print(records)
for record in records:
    # Converts the entire record into a Python dict
    record_dict = record.data()
#
for k in keys:
    print(f"\t{k}: {record_dict[k]}")

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")


['sender', 'coreValue']
[<Record sender='9998' coreValue=9>, <Record sender='9993' coreValue=4>, <Record sender='9991' coreValue=6>, <Record sender='9990' coreValue=8>, <Record sender='998' coreValue=11>, <Record sender='997' coreValue=14>, <Record sender='9965' coreValue=12>, <Record sender='9964' coreValue=10>, <Record sender='9962' coreValue=5>, <Record sender='996' coreValue=9>, <Record sender='9955' coreValue=11>, <Record sender='9952' coreValue=14>, <Record sender='995' coreValue=2>, <Record sender='9936' coreValue=2>, <Record sender='9933' coreValue=14>, <Record sender='993' coreValue=10>, <Record sender='9921' coreValue=4>, <Record sender='9915' coreValue=5>, <Record sender='9908' coreValue=3>, <Record sender='9905' coreValue=8>, <Record sender='9903' coreValue=6>, <Record sender='990' coreValue=6>, <Record sender='99' coreValue=9>, <Record sender='9892' coreValue=5>, <Record sender='9891' coreValue=5>, <Record sender='989' coreValue=10>, <Record sender='9883' coreValue=5>, <Re